<a href="https://colab.research.google.com/github/Husan2/NTU_GAI/blob/main/0311deepseek_benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deepseek（模型：Deepseek-V3)
# 第一步：設置Colab環境
- 首先，我們需要在Colab中設置環境來顯示互動式UI。

In [1]:
# 安裝必要套件
!pip install ipywidgets matplotlib

# 啟用Colab的小工具支援
from google.colab import output
output.enable_custom_widget_manager()

# 導入所需庫
import time
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import threading

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.0 MB/s eta 0:00:00


# 第二步：建立番茄鐘計時器基礎，導入所需庫和設定變數

In [2]:
# 番茄鐘時間設定
WORK_TIME = 25 * 60  # 工作時間25分鐘
SHORT_BREAK = 5 * 60  # 短休息5分鐘
LONG_BREAK = 15 * 60  # 長休息15分鐘

# 全局變數
pomodoro_count = 0
is_running = False
current_time = WORK_TIME
current_mode = "工作"
last_update_time = time.time()
timer_thread = None

# 第三步：計時器核心功能

In [3]:
# 註冊Python函數供JavaScript調用
def start_timer():
    global is_running, last_update_time, timer_thread
    if not is_running:
        is_running = True
        last_update_time = time.time()
        print("計時器已啟動!")
        # 啟動新的計時器線程
        timer_thread = threading.Thread(target=update_timer)
        timer_thread.daemon = True
        timer_thread.start()

def pause_timer():
    global is_running
    is_running = False
    print("計時器已暫停!")

def reset_timer():
    global current_time, is_running, current_mode, pomodoro_count
    is_running = False
    current_mode = "工作"
    current_time = WORK_TIME
    pomodoro_count = 0
    print("計時器已重置!")
    update_display()

def update_timer():
    global current_time, pomodoro_count, current_mode, is_running, last_update_time

    while is_running and current_time > 0:
        now = time.time()
        elapsed = now - last_update_time
        last_update_time = now

        current_time -= max(1, int(elapsed))
        update_display()
        time.sleep(1)

    if current_time <= 0 and is_running:
        handle_timer_completion()

def handle_timer_completion():
    global current_time, pomodoro_count, current_mode, is_running

    if current_mode == "工作":
        pomodoro_count += 1
        if pomodoro_count % 4 == 0:
            current_mode = "長休息"
            current_time = LONG_BREAK
        else:
            current_mode = "短休息"
            current_time = SHORT_BREAK
    else:
        current_mode = "工作"
        current_time = WORK_TIME

    # 播放提示音
    display(HTML('<audio autoplay><source src="https://www.soundjay.com/buttons/sounds/button-09.mp3" type="audio/mpeg"></audio>'))
    update_display()

    if is_running:
        update_timer()

def update_display():
    mins, secs = divmod(current_time, 60)
    time_str = f'{mins:02d}:{secs:02d}'
    display(HTML(f"""
    <script>
        document.getElementById('mode-display').innerText = '當前模式: {current_mode}';
        document.getElementById('timer-display').innerText = '{time_str}';
        document.getElementById('pomodoro-display').innerText = '已完成番茄鐘: {pomodoro_count}';
    </script>
    """))
    clear_output(wait=True)

# 註冊回調函數（必須放在函數定義之後）
output.register_callback('notebook_start_timer', start_timer)
output.register_callback('notebook_pause_timer', pause_timer)
output.register_callback('notebook_reset_timer', reset_timer)

# 第四步：建立UI元件

In [6]:
# 圖片URL
background_url = "https://imgcdn.cna.com.tw/www/WebPhotos/1024/20240225/2000x1333_wmkn_56954059532653_0.jpg"
cat_gif_url = "https://cdn.discordapp.com/attachments/725938412535808032/1353940562507595817/IMG_3343.gif?ex=67e37ae9&is=67e22969&hm=3bc6b2a8959c7e048a396efb26a09f95a9aca609298d052bb7f1847d98ad0a9a&"

# 創建介面HTML
interface_html = f"""
<div style="
    width: 400px;
    margin: 20px auto;
    font-family: 'Comic Sans MS', cursive, sans-serif;
    position: relative;
    border-radius: 15px;
    overflow: hidden;
    box-shadow: 0 4px 8px rgba(0,0,0,0.2);
">
    <!-- 背景圖層 -->
    <div style="
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        background-image: url('{background_url}');
        background-size: cover;
        opacity: 0.8;
        z-index: -1;
    "></div>

    <!-- 貓咪動畫 -->
    <div style="
        height: 120px;
        position: relative;
        overflow: hidden;
        background-color: rgba(255,255,255,0.3);
    ">
        <img src="{cat_gif_url}" style="
            position: absolute;
            bottom: 10px;
            left: 0;
            height: 80px;
            animation: walk 10s linear infinite;
        ">
    </div>

    <!-- 計時器內容 -->
    <div style="
        padding: 20px;
        background-color: rgba(255,255,255,0.7);
    ">
        <div id="mode-display" style="font-size: 20px; color: #4ecdc4; text-align: center; margin-bottom: 10px;">當前模式: 工作</div>
        <div id="timer-display" style="font-size: 48px; color: #ff6b6b; font-weight: bold; text-align: center; margin: 15px 0;">25:00</div>
        <div id="pomodoro-display" style="font-size: 20px; color: #ffbe76; text-align: center; margin-bottom: 15px;">已完成番茄鐘: 0</div>

        <!-- 按鈕區 -->
        <div style="display: flex; justify-content: center; gap: 10px;">
            <button onclick="google.colab.kernel.invokeFunction('notebook_start_timer', [], {{}})" style="
                padding: 8px 16px;
                background-color: #4CAF50;
                color: white;
                border: none;
                border-radius: 20px;
                cursor: pointer;
                font-family: 'Comic Sans MS', cursive, sans-serif;
            ">🐱 開始</button>

            <button onclick="google.colab.kernel.invokeFunction('notebook_pause_timer', [], {{}})" style="
                padding: 8px 16px;
                background-color: #FFC107;
                color: black;
                border: none;
                border-radius: 20px;
                cursor: pointer;
                font-family: 'Comic Sans MS', cursive, sans-serif;
            ">🐾 暫停</button>

            <button onclick="google.colab.kernel.invokeFunction('notebook_reset_timer', [], {{}})" style="
                padding: 8px 16px;
                background-color: #F44336;
                color: white;
                border: none;
                border-radius: 20px;
                cursor: pointer;
                font-family: 'Comic Sans MS', cursive, sans-serif;
            ">🙀 重置</button>
        </div>
    </div>
</div>

<style>
    @keyframes walk {{
        0% {{ left: -100px; }}
        100% {{ left: 100%; }}
    }}
</style>
"""

# 顯示介面
display(HTML(interface_html))
update_display()

# 第五步：添加自動更新機制

In [5]:
# 初始顯示
update_display()

# 顯示操作說明
print("貓咪番茄鐘已準備就緒！請使用下方按鈕控制：")
print("🐱 開始 - 啟動計時器")
print("🐾 暫停 - 暫停計時器")
print("🙀 重置 - 重置所有狀態")
print("----------------------------------")

貓咪番茄鐘已準備就緒！請使用下方按鈕控制：
🐱 開始 - 啟動計時器
🐾 暫停 - 暫停計時器
🙀 重置 - 重置所有狀態
----------------------------------
